In [ ]:
!hf auth login

In [ ]:
# !pip install protobuf

In [ ]:
# !pip install -q transformers=='4.46.1' numpy=='2.0.0' pydub ##gemma270m errors this
!pip install -q accelerate
!pip install -U -q bitsandbytes


In [ ]:
!pip install -q -U indic-nlp-library
!git clone https://github.com/anoopkunchukuttan/indic_nlp_resources.git

#for tts

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git

#for translation

In [ ]:
# Clone the github repository and navigate to the project directory.
!git clone https://github.com/AI4Bharat/IndicTrans2.git
%cd IndicTrans2/huggingface_interface

'''
NOTE: `%cd` used to make the change directory operation permanent.
'''

# # # Install all the dependencies and requirements associated with the project for running HF compatible models.
!source install.sh

#restart session after this !!!

In [ ]:
# !pip install -q -U transformers==4.55.4

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
import bitsandbytes, accelerate, transformers
bitsandbytes.__version__, accelerate.__version__, transformers.__version__

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", quantization_config=quantization_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")

In [ ]:
# chat = [
#     { "role": "user", "content": "write a 1min long bedtime story"},
# ]
# prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
# input_ids = prompt

# input_ids[0]
# story = model.generate(input_ids, max_new_tokens=500)
# story = tokenizer.decode(story[0], skip_special_tokens=True)
# print(story[story.find("model")+6:])

In [ ]:
# example input
prompt = "generate a short, fairy tale genre, nighttime story. the response should contain only the story."

inp = tokenizer(prompt, return_tensors="pt")['input_ids'].to(model.device)
output = model.generate(inp,
                        max_new_tokens=1000)


input_sentences = tokenizer.decode(output[0],
                                   skip_special_tokens=True,
                                   clean_up_tokenization_spaces=True).strip(prompt+"\n\n")

In [ ]:
input_sentences

In [ ]:
input_sentences = input_sentences.replace("**","").split("\n\n")

In [ ]:
input_sentences

#here translation

In [ ]:
input_sentences = [
    "When I was young, I used to go to the park every day.",
    "He has many old books, which he inherited from his ancestors.",
    "I can't figure out how to solve my problem.",
    "She is very hardworking and intelligent, which is why she got all the good marks.",
    "We watched a new movie last week, which was very inspiring.",
    "If you had met me at that time, we would have gone out to eat.",
    "She went to the market with her sister to buy a new sari.",
    "Raj told me that he is going to his grandmother's house next month.",
    "All the kids were having fun at the party and were eating lots of sweets.",
    "My friend has invited me to his birthday party, and I will give him a gift.",
]

In [ ]:
quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

# recommended to run this on a gpu with flash_attn installed
# don't set attn_implemetation if you don't have flash_attn

model_trans_name = "ai4bharat/indictrans2-en-indic-1B"

tokenizer_trans = AutoTokenizer.from_pretrained(model_trans_name, trust_remote_code=True)

model_trans = AutoModelForSeq2SeqLM.from_pretrained(
    model_trans_name,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=quantization_config,
).to(device)

model_trans.eval()

In [ ]:
src_lang, tgt_lang = "eng_Latn", "mal_Mlym"

ip = IndicProcessor(inference=True)
BATCH_SIZE = 100

translations = []
for i in range(0, len(input_sentences), BATCH_SIZE):
    batch = input_sentences[i : i + BATCH_SIZE]

    # Preprocess the batch and extract entity mappings
    batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

    # Tokenize the batch and generate input encodings
    inputs = tokenizer_trans(
        batch,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        return_attention_mask=True,
    ).to(device)

    # Generate translations using the model
    with torch.no_grad():
        generated_tokens = model_trans.generate(
            **inputs,
            use_cache=True,
            min_length=0,
            max_length=256,
            num_beams=5,
            num_return_sequences=1,
        )

    # Decode the generated tokens into text
    generated_tokens = tokenizer_trans.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    # Postprocess the translations, including entity replacement
    translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

    del inputs
    torch.cuda.empty_cache()

In [ ]:
translations = ''.join(translations)
translations

In [ ]:
story = translations

import nltk
from nltk.tokenize import sent_tokenize

from indicnlp import common
from indicnlp import loader

nltk.download('punkt')

# Set the path to the resources folder
INDIC_RESOURCES_PATH = "/content/indic_nlp_resources"

# Initialize the Indic NLP library
common.set_resources_path(INDIC_RESOURCES_PATH)
loader.load()

from indicnlp.tokenize import sentence_tokenize

def split_text(text):
    # Language code for Hindi
    lang = 'ml'
    sentences = sentence_tokenize.sentence_split(text, lang)
    return sentences

# Example usage
sentences = split_text(story)

chunks = []
current_chunk = ""
for sentence in sentences:
    if len(current_chunk) + len(sentence) <= 100:
        current_chunk += " " + sentence
    else:
        chunks.append(current_chunk.strip()+',')
        current_chunk = sentence
if current_chunk:
    chunks.append(current_chunk.strip()+',')

print(chunks)

#here tts

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, set_seed

model_tts = ParlerTTSForConditionalGeneration.from_pretrained("ai4bharat/indic-parler-tts").to(device)
tokenizer_tts = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")
description_tokenizer = AutoTokenizer.from_pretrained(model_tts.config.text_encoder._name_or_path)

prompt = chunks
description = len(chunks)*["Anjali's is telling a bedtime story and her voice is calm & soothing with gentle pacing and has no background noise."]

description_input_ids = description_tokenizer(description, return_tensors="pt", padding=True, add_special_tokens=True).to(device)
prompt_input_ids = tokenizer_tts(prompt, return_tensors="pt", padding=True, add_special_tokens=True).to(device)

set_seed(0)
generation = model_tts.generate(input_ids=description_input_ids.input_ids,
                            attention_mask=description_input_ids.attention_mask,
                            prompt_input_ids=prompt_input_ids.input_ids,
                            prompt_attention_mask=prompt_input_ids.attention_mask,
                            do_sample=True,
                            return_dict_in_generate=True)

In [ ]:
from pathlib import Path
from pydub import AudioSegment
import soundfile as sf

combined = AudioSegment.empty()

for i in range(len(chunks)):
    audio_arr = generation.sequences[i, :generation.audios_length[i]].cpu().numpy().squeeze()
    temp_file = f"/content/temp.wav"
    sf.write(temp_file, audio_arr.astype(float), model_tts.config.sampling_rate)

    audio_segment = AudioSegment.from_wav(temp_file)
    combined += audio_segment

    Path(temp_file).unlink()

combined.export("output.wav", format="wav")

In [ ]:
from IPython.display import Audio, display

# Play the audio
Audio("output.wav", rate=model_tts.config.sampling_rate) # Autoplay=True plays it automatically

In [ ]:
!pip uninstall transformers
!pip install -q -U transformers

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
import transformers
print(transformers.__version__)